# **TASK 1 — Problem & Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Apache Spark uses Java, so first we must install that
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-8-jdk-headless -qq

# Unpack Spark from google drive
!tar xzf /content/drive/MyDrive/spark-3.3.0-bin-hadoop3.tgz

# Set up environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.3.0-bin-hadoop3"

# Install findspark, which helps python locate the psyspark module files
!pip install -q findspark
import findspark
findspark.init()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .../openjdk-8-jre-headless_8u49

In [ ]:
# Finally, we initialse a "SparkSession", which handles the computations
from pyspark.sql import SparkSession
spark = SparkSession.builder\
        .master("local")\
        .appName("Colab")\
        .config('spark.ui.port', '4050')\
        .getOrCreate()

# pyspark.sql.functions countains all the transformations and actions you will
# need

from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [ ]:
# 1. Download the Heterogeneity Activity Recognition dataset
!wget -q https://archive.ics.uci.edu/static/public/344/heterogeneity+activity+recognition.zip -O har_hetero.zip

# 2. Unzip the main download
!unzip -q -o har_hetero.zip -d har_hetero_extracted

# 3. Unzip the nested zip files found inside the extracted folder
!unzip -q -o "/content/har_hetero_extracted/Activity recognition exp.zip" -d /content/har_hetero_extracted/data
!unzip -q -o "/content/har_hetero_extracted/Still exp.zip" -d /content/har_hetero_extracted/data

# 4. Load the dataset using PySpark with recursive lookup
har_hetero_df = spark.read\
    .option('header', 'True')\
    .option('inferSchema', 'True')\
    .option('recursiveFileLookup', 'True')\
    .csv('/content/har_hetero_extracted/data/')

# Show results
har_hetero_df.printSchema()
har_hetero_df.show(20)

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- z: string (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)

+-----+-------------+-------------------+--------------------+--------------------+--------------------+----+------+--------+-----+
|Index| Arrival_Time|      Creation_Time|                   x|                   y|                   z|User| Model|  Device|   gt|
+-----+-------------+-------------------+--------------------+--------------------+--------------------+----+------+--------+-----+
|    0|1424696633909|1424696631914042029|         0.013748169|-0.00062561035000...|        -0.023376465|   a|nexus4|nexus4_1|stand|
|    1|1424696633909|1424696631919046912|0.014816283999999999|       -0.0016937256|         -0.0

In [ ]:
har_hetero_df.printSchema()

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- z: string (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)



In [ ]:
!ls -lh har_hetero.zip /content/har_hetero_extracted/

-rw-r--r-- 1 root root 785M Jun 14 04:01 har_hetero.zip

/content/har_hetero_extracted/:
total 785M
-rwx------ 1 root root 742M May 22  2023 'Activity recognition exp.zip'
drwxr-xr-x 5 root root 4.0K Jun 14 04:02  data
-rwx------ 1 root root  43M May 22  2023 'Still exp.zip'
